# LUAD Data Preprocessing Pipeline
Builds node features (HPA, GDC mutation/CNV), STRING PPI network, conventional network topology features, and GDA labels.


In [2]:
pip install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 16.1 MB/s eta 0:00:00 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
import igraph as ig  # pip install python-igraph
from collections import Counter
from node2vec import Node2Vec
import networkx as nx


In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
12.8
True
NVIDIA GeForce RTX 5060


In [3]:
def remove_redun(el, verbose=False):
    if verbose:
        print("Original Size: ", len(el))

    el_new = el.iloc[:, 0:2].apply(sorted, axis=1)
    el_new = pd.DataFrame.from_dict(dict(zip(el_new.index, el_new.values))).T
    el_new = el_new.drop_duplicates()
    if verbose:
        postDrop = len(el_new)
        print("After Dropping Duplicates: ", len(el_new), "(-", len(el)-postDrop, ")")

    el_new = el_new.merge(el, left_on=[el_new.columns[0], el_new.columns[1]],
                          right_on=[el.columns[0], el.columns[1]])
    if verbose:
        print("After Merging: ", len(el_new), "(-", postDrop-len(el_new), ")")
        print()

    return el_new.iloc[:, 2:]


def map_IDs(el, gmap, verbose=False, dropNaNvalues=True):
    gp_map_f = gmap.set_index('#string_protein_id')['alias']
    el_converted = el.reset_index(drop=True)

    el_converted[el_converted.columns[0]] = el_converted[el_converted.columns[0]].map(gp_map_f)
    el_converted[el_converted.columns[1]] = el_converted[el_converted.columns[1]].map(gp_map_f)

    if verbose:
        print("NaN values per Column:",
              el_converted[el_converted.columns[0]].isna().sum(),
              el_converted[el_converted.columns[1]].isna().sum())

    if dropNaNvalues:
        el_converted = el_converted.dropna()
        if verbose:
            print("New edge list size:", len(el_converted), "( -", len(el)-len(el_converted), ")")

    return el_converted


def ImportSTRING():
    el_map = pd.read_csv(
        r"9606.protein.aliases.v12.0.txt",
        sep="\t"
    )
    el = pd.read_csv(
        r"9606.protein.links.v12.0_sc.txt",
        sep=" "
    )
    el_map = el_map.loc[el_map.source == 'Ensembl_gene']

    el = remove_redun(el, True)
    el = map_IDs(el, el_map, verbose=True)

    return el

In [41]:

def ImportDGN():
    dgn = pd.read_csv("gda_LUAD.csv")
    dgn_dict = pd.read_csv(
        "gda_dictionary.csv",
        index_col=None
    )

    score_threshold = 0.6  
    ei_threshold    = 0.5

    dgn = dgn[['Gene_Symbol', 'Ei', 'Score']]
    dgn = dgn.loc[dgn['Score'] >= score_threshold]
    dgn = dgn.loc[dgn['Ei'] > ei_threshold]
    dgn.rename({'Score': 'gda_score'}, axis=1, inplace=True)
    dgn = dgn.merge(dgn_dict, on="Gene_Symbol").drop(['Gene_Symbol'], axis=1)
    dgn['gda_score'] = 1

    return dgn[['ensembl', 'gda_score']]

In [9]:
def ImportHPA():
    hpa = pd.read_csv(
        r"lung.tsv",
        sep='\t'
    ).drop_duplicates(subset='Gene')

    identifiers = ["Gene", "Ensembl"]
    discrete_features = [
        "Protein class", "Biological process", "Molecular function",
        "Disease involvement", "Subcellular location",
    ]

    hpa_features = hpa.loc[:, hpa.columns.isin(identifiers + discrete_features)]

    # normalise continuous features

    def explode(feature):
        return feature.apply(lambda x: x.replace(' ', '').split(','))

    hpa_clean = hpa.fillna('')
    for ft in discrete_features:
        hpa_clean[ft] = explode(hpa_clean[ft])

    protein_class        = hpa_clean["Protein class"].explode().unique()
    biological_process   = hpa_clean["Biological process"].explode().unique()
    molecular_function   = hpa_clean["Molecular function"].explode().unique()
    disease_involvement  = hpa_clean["Disease involvement"].explode().unique()
    subcellular_location = hpa_clean["Subcellular location"].explode().unique()

    GO_features = np.concatenate([
        protein_class, biological_process, molecular_function,
        disease_involvement, subcellular_location
    ])

    RowFeatures = pd.DataFrame(data=0, index=hpa_clean['Ensembl'], columns=GO_features)
    counter = 0
    for index, row in RowFeatures.iterrows():
        features = hpa_clean.iloc[counter][
            ['Protein class', 'Biological process', 'Molecular function',
             'Disease involvement', 'Subcellular location']
        ].to_list()
        flattened = [item for sublist in features for item in sublist if item]
        for t in flattened:
            row[t] = 1
        counter += 1

    # truncated SVD to 200 dims
    n_comp    = 200
    svd       = TruncatedSVD(n_components=n_comp)
    svdModel  = svd.fit(RowFeatures)
    visits_emb = svdModel.transform(RowFeatures)

    hpa_reduced = pd.DataFrame(data=visits_emb, index=RowFeatures.index).reset_index(names="Ensembl")

    hpa_final = hpa_reduced

    hpa_final.columns = ['hpa_' + str(col) for col in hpa_final.columns]
    hpa_final = hpa_final.rename({
        'hpa_Ensembl': 'ensembl',
    }, axis=1)

    return hpa_final

In [22]:
def ImportGDC():
    gdc = pd.read_csv("CNVs_LUAD.tsv", sep='\t')

    gdc = gdc.rename(columns={
        'ensembl': 'ensembl',
        'cohort_ssm_affected_cases_percentage': 'nih_ssm_in_cohort',
        'gdc_ssm_affected_cases_percentage':    'nih_ssm_across_gdc',
        'cohort_cnv_gain_cases_percentage':     'nih_cnv_gain',
        'num_mutations':                        'nih_tot_mutations',
    })

    gdc['nih_cnv_loss'] = gdc['cohort_cnv_homozygous_deletion_cases_percentage']

    for col in ['nih_ssm_in_cohort', 'nih_ssm_across_gdc', 'nih_cnv_gain', 'nih_cnv_loss']:
        gdc[col] = gdc[col] / 100.0

    gdc = gdc[['ensembl', 'nih_ssm_in_cohort', 'nih_ssm_across_gdc',
               'nih_cnv_gain', 'nih_cnv_loss', 'nih_tot_mutations']]

    return gdc

In [7]:
import os

def Runnode2vec(filepath):
    df = pd.read_csv(filepath, sep="\t", header=None, names=["source", "target", "weight"])
    df["weight"] /= 1000  # normalise STRING combined_score to [0,1]

    G = nx.from_pandas_edgelist(df, "source", "target", ["weight"], create_using=nx.Graph())
    n_workers = max(1, min(4, os.cpu_count() or 1))

    node2vec = Node2Vec(
        G, dimensions=128, walk_length=60, num_walks=15,
        workers=n_workers, p=1, q=0.5,
        quiet=True,         
    )
    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    emd = pd.DataFrame([model.wv[str(node)] for node in G.nodes()], index=G.nodes())
    emd.columns = [f'network_{i}' for i in range(emd.shape[1])]

    return emd.reset_index().rename(columns={"index": "ensembl"})

## Run pipeline

In [10]:
hpa = ImportHPA()
hpa

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_190,hpa_191,hpa_192,hpa_193,hpa_194,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.004175,-0.022976,-0.050481,0.017768,-0.037165,0.025103,-0.024738,0.003225,0.002854,0.030352
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.014279,0.052805,0.011561,0.003775,-0.005723,-0.003326,-0.070767,-0.056659,0.049136,-0.028455
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.023961,-0.005237,0.013972,0.004382,-0.004958,0.003655,-0.037395,0.008726,-0.017717,0.014048
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,0.121729,-0.001900,0.016658,0.003927,0.008292,0.083609,0.113913,0.046002,-0.101680,0.065675
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,-0.013397,-0.001437,0.027928,0.009143,-0.003394,0.000864,-0.045981,-0.040801,0.019711,-0.015938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14337,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,-0.007305,0.001132,0.005011,-0.001115,-0.000903,0.009542,-0.000347,0.015048,0.000995,0.007774
14338,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,-0.003713,-0.001763,-0.020740,-0.013851,-0.014466,0.005598,-0.011136,-0.010212,0.014063,-0.004217
14339,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,-0.002443,-0.003263,-0.003034,-0.007033,-0.016503,0.004853,-0.016363,0.011387,0.001540,-0.005837
14340,ENSG00000074755,0.760256,-0.722286,0.580587,0.545036,1.117914,-0.245676,0.345980,0.324863,0.193817,...,-0.008475,0.010132,0.009448,-0.001056,0.007613,-0.003365,-0.000388,0.003167,0.007667,0.002589


In [11]:
el = ImportSTRING()

print(f'Edges before filtering: {len(el):,}')
el = el.loc[el['combined_score'] > 700].reset_index(drop=True)
print(f'Edges after removing score >= 400: {len(el):,}')

el

Original Size:  13715404
After Dropping Duplicates:  6857702 (- 6857702 )
After Merging:  6857702 (- 0 )

NaN values per Column: 0 0
New edge list size: 6857702 ( - 0 )
Edges before filtering: 6,857,702
Edges after removing score >= 400: 236,000


,protein1,protein2,combined_score
0,ENSG00000004059,ENSG00000072818,825
1,ENSG00000004059,ENSG00000122218,718
2,ENSG00000004059,ENSG00000090565,952
3,ENSG00000004059,ENSG00000184432,752
4,ENSG00000004059,ENSG00000105669,795
...,...,...,...
235995,ENSG00000143933,ENSG00000070808,962
235996,ENSG00000143933,ENSG00000145335,918
235997,ENSG00000162434,ENSG00000051382,933
235998,ENSG00000070808,ENSG00000121281,707


In [26]:
master = hpa
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_190,hpa_191,hpa_192,hpa_193,hpa_194,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.004175,-0.022976,-0.050481,0.017768,-0.037165,0.025103,-0.024738,0.003225,0.002854,0.030352
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.014279,0.052805,0.011561,0.003775,-0.005723,-0.003326,-0.070767,-0.056659,0.049136,-0.028455
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.023961,-0.005237,0.013972,0.004382,-0.004958,0.003655,-0.037395,0.008726,-0.017717,0.014048
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,0.121729,-0.001900,0.016658,0.003927,0.008292,0.083609,0.113913,0.046002,-0.101680,0.065675
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,-0.013397,-0.001437,0.027928,0.009143,-0.003394,0.000864,-0.045981,-0.040801,0.019711,-0.015938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14337,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,-0.007305,0.001132,0.005011,-0.001115,-0.000903,0.009542,-0.000347,0.015048,0.000995,0.007774
14338,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,-0.003713,-0.001763,-0.020740,-0.013851,-0.014466,0.005598,-0.011136,-0.010212,0.014063,-0.004217
14339,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,-0.002443,-0.003263,-0.003034,-0.007033,-0.016503,0.004853,-0.016363,0.011387,0.001540,-0.005837
14340,ENSG00000074755,0.760256,-0.722286,0.580587,0.545036,1.117914,-0.245676,0.345980,0.324863,0.193817,...,-0.008475,0.010132,0.009448,-0.001056,0.007613,-0.003365,-0.000388,0.003167,0.007667,0.002589


In [27]:
gdc=ImportGDC()
gdc

,ensembl,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000155657,0.5081,0.2901,0.2172,0.0,825
1,ENSG00000141510,0.5027,0.3137,0.0489,0.0,208
2,ENSG00000181143,0.4436,0.1849,0.0333,0.0,505
3,ENSG00000198626,0.4204,0.1305,0.3933,0.0,434
4,ENSG00000164796,0.4132,0.1315,0.3014,0.0,409
...,...,...,...,...,...,...
20363,ENSG00000288660,0.0000,0.0000,0.2270,0.0,0
20364,ENSG00000288669,0.0000,0.0000,0.0313,0.0,0
20365,ENSG00000288671,0.0000,0.0000,0.1311,0.0,0
20366,ENSG00000288674,0.0000,0.0000,0.4031,0.0,0


In [28]:
master = master.merge(gdc, on='ensembl')
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.025103,-0.024738,0.003225,0.002854,0.030352,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.003326,-0.070767,-0.056659,0.049136,-0.028455,0.0000,0.0066,0.0881,0.0,0
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.003655,-0.037395,0.008726,-0.017717,0.014048,0.0018,0.0072,0.1840,0.0,1
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,0.083609,0.113913,0.046002,-0.101680,0.065675,0.0268,0.0101,0.1683,0.0,16
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,0.000864,-0.045981,-0.040801,0.019711,-0.015938,0.0125,0.0075,0.1859,0.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14176,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,0.009542,-0.000347,0.015048,0.000995,0.007774,0.0143,0.0101,0.1898,0.0,10
14177,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,0.005598,-0.011136,-0.010212,0.014063,-0.004217,0.0197,0.0093,0.1722,0.0,11
14178,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,0.004853,-0.016363,0.011387,0.001540,-0.005837,0.0125,0.0080,0.2505,0.0,8
14179,ENSG00000074755,0.760256,-0.722286,0.580587,0.545036,1.117914,-0.245676,0.345980,0.324863,0.193817,...,-0.003365,-0.000388,0.003167,0.007667,0.002589,0.0411,0.0260,0.0548,0.0,24


In [29]:
list_of_dropped_genes=pd.read_csv("19k_LUAD_mean_expression_ensembl.csv")
master_590 = master[
    master["ensembl"].isin(list_of_dropped_genes["ensembl"])
]
master=master_590
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.025103,-0.024738,0.003225,0.002854,0.030352,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.003326,-0.070767,-0.056659,0.049136,-0.028455,0.0000,0.0066,0.0881,0.0,0
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.003655,-0.037395,0.008726,-0.017717,0.014048,0.0018,0.0072,0.1840,0.0,1
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,0.083609,0.113913,0.046002,-0.101680,0.065675,0.0268,0.0101,0.1683,0.0,16
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,0.000864,-0.045981,-0.040801,0.019711,-0.015938,0.0125,0.0075,0.1859,0.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14176,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,0.009542,-0.000347,0.015048,0.000995,0.007774,0.0143,0.0101,0.1898,0.0,10
14177,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,0.005598,-0.011136,-0.010212,0.014063,-0.004217,0.0197,0.0093,0.1722,0.0,11
14178,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,0.004853,-0.016363,0.011387,0.001540,-0.005837,0.0125,0.0080,0.2505,0.0,8
14179,ENSG00000074755,0.760256,-0.722286,0.580587,0.545036,1.117914,-0.245676,0.345980,0.324863,0.193817,...,-0.003365,-0.000388,0.003167,0.007667,0.002589,0.0411,0.0260,0.0548,0.0,24


In [30]:
master = master.reset_index(drop=True)
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.025103,-0.024738,0.003225,0.002854,0.030352,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.003326,-0.070767,-0.056659,0.049136,-0.028455,0.0000,0.0066,0.0881,0.0,0
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.003655,-0.037395,0.008726,-0.017717,0.014048,0.0018,0.0072,0.1840,0.0,1
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,0.083609,0.113913,0.046002,-0.101680,0.065675,0.0268,0.0101,0.1683,0.0,16
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,0.000864,-0.045981,-0.040801,0.019711,-0.015938,0.0125,0.0075,0.1859,0.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13444,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,0.009542,-0.000347,0.015048,0.000995,0.007774,0.0143,0.0101,0.1898,0.0,10
13445,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,0.005598,-0.011136,-0.010212,0.014063,-0.004217,0.0197,0.0093,0.1722,0.0,11
13446,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,0.004853,-0.016363,0.011387,0.001540,-0.005837,0.0125,0.0080,0.2505,0.0,8
13447,ENSG00000074755,0.760256,-0.722286,0.580587,0.545036,1.117914,-0.245676,0.345980,0.324863,0.193817,...,-0.003365,-0.000388,0.003167,0.007667,0.002589,0.0411,0.0260,0.0548,0.0,24


In [31]:
el_allgenes = pd.concat([el['protein1'], el['protein2']]).drop_duplicates()
master = master.loc[master['ensembl'].isin(el_allgenes)]
print(f'Genes after STRING intersection: {len(master)}')
master

Genes after STRING intersection: 11648


,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.025103,-0.024738,0.003225,0.002854,0.030352,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.003326,-0.070767,-0.056659,0.049136,-0.028455,0.0000,0.0066,0.0881,0.0,0
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.003655,-0.037395,0.008726,-0.017717,0.014048,0.0018,0.0072,0.1840,0.0,1
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,0.083609,0.113913,0.046002,-0.101680,0.065675,0.0268,0.0101,0.1683,0.0,16
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,0.000864,-0.045981,-0.040801,0.019711,-0.015938,0.0125,0.0075,0.1859,0.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13442,ENSG00000198205,1.239520,-1.417517,0.776085,0.028084,0.647712,-0.197421,-0.058612,-0.142415,0.021137,...,0.006221,-0.001070,0.013464,-0.000321,0.007117,0.0179,0.0125,0.0000,0.0,10
13444,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,0.009542,-0.000347,0.015048,0.000995,0.007774,0.0143,0.0101,0.1898,0.0,10
13445,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,0.005598,-0.011136,-0.010212,0.014063,-0.004217,0.0197,0.0093,0.1722,0.0,11
13446,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,0.004853,-0.016363,0.011387,0.001540,-0.005837,0.0125,0.0080,0.2505,0.0,8


In [32]:
el_intersect = (
    el.iloc[:, :3]
    .merge(master["ensembl"], right_on="ensembl", left_on='protein1')
    .drop("ensembl", axis=1)
)
el_intersect = (
    el_intersect
    .merge(master["ensembl"], right_on="ensembl", left_on='protein2')
    .drop("ensembl", axis=1)
    .rename(columns={'protein1': 'gene1', 'protein2': 'gene2'})
)

el = el_intersect.merge(el, right_on=['protein1', 'protein2'], left_on=['gene1', 'gene2']).drop(['protein1', 'protein2'], axis=1)
el

,gene1,gene2,combined_score_x,combined_score_y
0,ENSG00000004059,ENSG00000072818,825,825
1,ENSG00000004059,ENSG00000122218,718,718
2,ENSG00000004059,ENSG00000090565,952,952
3,ENSG00000004059,ENSG00000184432,752,752
4,ENSG00000004059,ENSG00000105669,795,795
...,...,...,...,...
162574,ENSG00000198668,ENSG00000145335,973,973
162575,ENSG00000274211,ENSG00000162434,786,786
162576,ENSG00000143933,ENSG00000145335,918,918
162577,ENSG00000162434,ENSG00000051382,933,933


In [33]:
el[['gene1', 'gene2', 'combined_score_x']].to_csv(
    '19k_luad.edg', index=False, header=False, sep='\t'
)

In [34]:
df = pd.read_csv(
    "19k_luad.edg",
    sep="\t", header=None, names=["source", "target", "weight"]
)
print(df.head())
print('NaN weights:', df["weight"].isna().sum())

            source           target  weight
0  ENSG00000004059  ENSG00000072818     825
1  ENSG00000004059  ENSG00000122218     718
2  ENSG00000004059  ENSG00000090565     952
3  ENSG00000004059  ENSG00000184432     752
4  ENSG00000004059  ENSG00000105669     795
NaN weights: 0


In [35]:
network = Runnode2vec("19k_luad.edg")
master = master.merge(network, on='ensembl')
master.to_csv("node_node2vec_data_latest.csv", index=None)
master

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,network_118,network_119,network_120,network_121,network_122,network_123,network_124,network_125,network_126,network_127
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.327350,-0.205753,0.182960,-0.550097,0.074443,-0.320523,-0.287383,0.156345,0.037887,-0.057937
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.108672,-0.462063,0.315828,-0.349039,-0.348705,0.029682,-0.822822,-0.128510,0.107428,0.001755
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.162144,-0.161835,0.122013,0.252953,0.221002,0.188961,0.171036,-0.074615,0.036719,-0.176683
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,-0.228740,-0.204287,-0.556206,-0.100686,0.511818,0.128199,0.043797,-0.365749,-0.105148,0.041208
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,0.331550,-0.256717,0.583400,-0.685297,0.181385,-0.092898,-0.223120,0.138173,-0.843833,-0.015200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11442,ENSG00000198205,1.239520,-1.417517,0.776085,0.028084,0.647712,-0.197421,-0.058612,-0.142415,0.021137,...,-0.204936,-0.346440,-0.200723,-0.386905,0.067715,-0.267302,0.401131,0.054083,-0.256946,0.308465
11443,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,-0.224528,-0.289764,-0.099868,-0.312187,0.077197,-0.185213,0.296417,-0.051232,-0.231659,0.160015
11444,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,0.280172,0.041160,-0.227607,-0.286642,-0.078675,0.177313,-0.079360,-0.356234,-0.014635,0.050656
11445,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,0.548837,0.200505,-0.083219,-0.162736,-0.361646,-0.174495,0.050778,-0.067854,-0.329535,0.054119


In [36]:
master.to_csv("node_networkfeatures_19k_cnv.csv", index=False)

In [37]:
df = pd.read_csv(
    "19k_luad.edg",
    sep="\t", header=None, names=["source", "target", "weight"]
)
print(df.head())
print('NaN weights:', df["weight"].isna().sum())

            source           target  weight
0  ENSG00000004059  ENSG00000072818     825
1  ENSG00000004059  ENSG00000122218     718
2  ENSG00000004059  ENSG00000090565     952
3  ENSG00000004059  ENSG00000184432     752
4  ENSG00000004059  ENSG00000105669     795
NaN weights: 0


In [38]:
edge_array = df[["source", "target", "weight"]].to_numpy()
np.save("19k_luad.npy", edge_array, allow_pickle=True)
print("Saved edge_list_latest1.npy with shape:", edge_array.shape)

loaded = np.load("19k_luad.npy", allow_pickle=True)
print(loaded[:5])

Saved edge_list_latest1.npy with shape: (162579, 3)
[['ENSG00000004059' 'ENSG00000072818' 825]
 ['ENSG00000004059' 'ENSG00000122218' 718]
 ['ENSG00000004059' 'ENSG00000090565' 952]
 ['ENSG00000004059' 'ENSG00000184432' 752]
 ['ENSG00000004059' 'ENSG00000105669' 795]]


In [39]:
master=pd.read_csv("node_networkfeatures_19k_cnv.csv")
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,network_118,network_119,network_120,network_121,network_122,network_123,network_124,network_125,network_126,network_127
0,ENSG00000175899,0.428712,0.166294,0.098041,0.123415,-0.094024,1.744920,-0.167404,0.377257,0.053228,...,0.327350,-0.205753,0.182960,-0.550097,0.074443,-0.320523,-0.287383,0.156345,0.037887,-0.057937
1,ENSG00000128274,0.680279,0.633793,-1.017946,-0.415402,1.443136,-0.197629,0.217684,-0.075812,0.083857,...,-0.108673,-0.462063,0.315828,-0.349039,-0.348705,0.029682,-0.822822,-0.128510,0.107428,0.001755
2,ENSG00000094914,2.392024,0.827001,0.227754,0.292725,-0.359296,-1.082123,0.388393,0.276827,0.760253,...,0.162144,-0.161835,0.122013,0.252953,0.221002,0.188961,0.171036,-0.074615,0.036719,-0.176683
3,ENSG00000081760,0.627875,0.569793,-0.890085,-0.024347,1.202415,-0.099912,0.072496,-0.137409,-0.069719,...,-0.228740,-0.204287,-0.556206,-0.100686,0.511818,0.128199,0.043797,-0.365749,-0.105148,0.041208
4,ENSG00000114771,1.063872,0.069031,-0.936957,-0.822382,0.584402,-0.108200,-0.437454,-0.319055,0.016982,...,0.331550,-0.256717,0.583400,-0.685297,0.181385,-0.092898,-0.223120,0.138173,-0.843833,-0.015200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11442,ENSG00000198205,1.239520,-1.417517,0.776085,0.028084,0.647712,-0.197421,-0.058612,-0.142415,0.021137,...,-0.204936,-0.346440,-0.200723,-0.386905,0.067715,-0.267302,0.401131,0.054083,-0.256946,0.308465
11443,ENSG00000070476,0.988306,-1.106771,0.687603,0.021939,0.563369,-0.004585,-0.274047,-0.846450,0.131344,...,-0.224528,-0.289764,-0.099868,-0.312187,0.077197,-0.185213,0.296417,-0.051232,-0.231659,0.160015
11444,ENSG00000162378,0.858317,-0.264733,-0.426122,0.242468,-0.121435,-0.018673,-0.534922,-0.361570,-0.584347,...,0.280172,0.041160,-0.227607,-0.286642,-0.078675,0.177313,-0.079360,-0.356234,-0.014635,0.050656
11445,ENSG00000159840,0.963116,-0.195811,-0.399642,0.324629,-0.375629,0.654206,-0.471037,-0.204208,0.094502,...,0.548837,0.200505,-0.083219,-0.162736,-0.361646,-0.174495,0.050778,-0.067854,-0.329535,0.054119


In [42]:
dgn = ImportDGN()
dgn = dgn.loc[dgn['ensembl'].isin(master['ensembl'])]
print(f'LIHC GDA genes overlapping final master: {len(dgn)}')
dgn

LIHC GDA genes overlapping final master: 108


,ensembl,gda_score
1,ENSG00000133703,1
2,ENSG00000146648,1
3,ENSG00000157764,1
4,ENSG00000141510,1
5,ENSG00000047936,1
...,...,...
118,ENSG00000177169,1
119,ENSG00000179218,1
123,ENSG00000140538,1
124,ENSG00000131023,1


In [43]:
master["gda_score"] = np.nan
master.loc[master["ensembl"].isin(dgn["ensembl"]), "gda_score"] = 1

num_ones = (master["gda_score"] == 1).sum()
print("Number of positive (gda_score=1) genes:", num_ones)

Number of positive (gda_score=1) genes: 108


In [44]:
master.to_csv("200_19k_cnv.csv", index=None)
print('Saved: 200_node_network_features_latest.csv')

Saved: 200_node_network_features_latest.csv
